## Example 1: MypycBuildHook Initialization and Configuration

Demonstrates initializing the MypycBuildHook class and accessing its configuration.

In [ ]:
# Example 1: MypycBuildHook initialization and configuration
import tempfile
import textwrap
from pathlib import Path

try:
    from hatch_mypyc.plugin import MypycBuildHook
    
    # Create temporary project directory
    with tempfile.TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)
        
        # Create minimal pyproject.toml
        pyproject = root / "pyproject.toml"
        pyproject.write_text(textwrap.dedent("""
        [build-system]
        requires = ["hatchling", "hatch-mypyc"]
        build-backend = "hatchling.build"
        
        [project]
        name = "test-pkg"
        version = "0.1.0"
        """))
        
        # Create build hook instance with custom config
        config = {
            'mypy-args': ['--strict', '--ignore-missing-imports'],
            'options': {
                'separate': True,
                'opt_level': '3'
            },
            'include': ['src/**/*.py'],
            'exclude': ['tests/**/*.py', '**/*_test.py']
        }
        
        # Initialize MypycBuildHook
        hook = MypycBuildHook(
            str(root),
            config,
            {},
            None,
            str(root),
            'wheel',
            None
        )
        
        # Verify configuration was loaded
        print(f'MypycBuildHook initialized successfully')
        print(f'Hook class: {hook.__class__.__name__}')
        print(f'Root directory: {hook.root}')
        print(f'Build directory exists: {hasattr(hook, "build_directory")}')
        print(f'Config attributes: {[attr for attr in dir(hook) if not attr.startswith("_")][:8]}')
        print('Example 1: MypycBuildHook initialization passed')
        
except Exception as e:
    print(f'Example 1 failed: {e}')
    raise AssertionError(
        f"hatch_mypyc example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: File Inclusion/Exclusion with Pattern Matching

Tests the include_path method for filtering files based on patterns.

In [ ]:
# Example 2: File inclusion/exclusion with include_path method
import tempfile
import textwrap
from pathlib import Path

try:
    from hatch_mypyc.plugin import MypycBuildHook
    
    with tempfile.TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)
        
        # Create test files
        src_dir = root / "src" / "mypackage"
        src_dir.mkdir(parents=True)
        (src_dir / "__init__.py").touch()
        (src_dir / "core.py").touch()
        (src_dir / "utils.py").touch()
        
        tests_dir = root / "tests"
        tests_dir.mkdir()
        (tests_dir / "test_core.py").touch()
        
        # Configure hook with include/exclude patterns
        config = {
            'include': ['src/**/*.py'],
            'exclude': ['**/__init__.py', 'tests/**/*.py']
        }
        
        hook = MypycBuildHook(str(root), config, {}, None, str(root), 'wheel', None)
        
        # Test paths against patterns
        test_paths = [
            'src/mypackage/__init__.py',
            'src/mypackage/core.py',
            'src/mypackage/utils.py',
            'tests/test_core.py'
        ]
        
        print('Testing file inclusion/exclusion patterns:')
        for path in test_paths:
            # Use hook's include_path method
            should_include = hook.include_path(path)
            status = 'INCLUDED' if should_include else 'EXCLUDED'
            print(f'  {path}: {status}')
        
        # Verify expected behavior
        assert hook.include_path('src/mypackage/core.py'), 'core.py should be included'
        assert not hook.include_path('src/mypackage/__init__.py'), '__init__.py should be excluded'
        assert not hook.include_path('tests/test_core.py'), 'test files should be excluded'
        
        print('Example 2: File inclusion/exclusion test passed')
        
except Exception as e:
    print(f'Example 2 failed: {e}')
    raise AssertionError(
        f"hatch_mypyc example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Configuration Properties Access

Demonstrates accessing configuration properties unique to MypycBuildHook.

In [ ]:
# Example 3: Configuration properties access
import tempfile
from pathlib import Path

try:
    from hatch_mypyc.plugin import MypycBuildHook
    
    with tempfile.TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)
        
        # Create hook with comprehensive configuration
        config = {
            'mypy-args': ['--strict', '--ignore-missing-imports', '--no-warn-unused-ignores'],
            'options': {
                'separate': True,
                'opt_level': '3',
                'multi_file': True
            },
            'include': ['src/**/*.py', 'lib/**/*.py'],
            'exclude': ['**/__init__.py', '**/test_*.py']
        }
        
        hook = MypycBuildHook(str(root), config, {}, None, str(root), 'wheel')
        
        # Access config properties (unique to MypycBuildHook)
        print('Configuration properties:')
        
        # config_mypy_args property
        mypy_args = hook.config_mypy_args
        print(f'  config_mypy_args: {mypy_args}')
        
        # config_options property
        options = hook.config_options
        print(f'  config_options: {options}')
        
        # config_separation property
        separation = hook.config_separation
        print(f'  config_separation: {separation}')
        
        # config_include property
        include = hook.config_include
        print(f'  config_include: {include}')
        
        # config_exclude property
        exclude = hook.config_exclude
        print(f'  config_exclude: {exclude}')
        
        # Verify configurations were properly loaded
        assert len(mypy_args) == 3, 'Should have 3 mypy args'
        assert separation is True, 'Separation should be enabled'
        assert 'opt_level' in options, 'Options should contain opt_level'
        assert len(include) == 2, 'Should have 2 include patterns'
        assert len(exclude) == 2, 'Should have 2 exclude patterns'
        
        print('Example 3: Configuration properties test passed')
        
except Exception as e:
    print(f'Example 3 failed: {e}')
    raise AssertionError(
        f"hatch_mypyc example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: Mypyc Compilation Options Configuration

Demonstrates configuring Mypyc compilation options through the hook.

In [ ]:
# Example 4: Mypyc compilation options configuration
import tempfile
import textwrap
from pathlib import Path

try:
    from hatch_mypyc.plugin import MypycBuildHook
    
    with tempfile.TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)
        
        # Test different option configurations
        test_configs = [
            {
                'name': 'Basic optimization',
                'config': {
                    'options': {'opt_level': '3'}
                }
            },
            {
                'name': 'Separate compilation',
                'config': {
                    'options': {'separate': True}
                }
            },
            {
                'name': 'Custom build directory',
                'config': {
                    'options': {'build-dir': 'custom_build'}
                }
            },
            {
                'name': 'Combined options',
                'config': {
                    'options': {
                        'opt_level': '3',
                        'separate': True,
                        'multi_file': True
                    },
                    'mypy-args': ['--strict', '--no-warn-unused-ignores']
                }
            }
        ]
        
        print('Testing Mypyc compilation option configurations:')
        for test in test_configs:
            try:
                hook = MypycBuildHook(
                    str(root),
                    test['config'],
                    {},
                    None,
                    str(root),
                    'wheel',
                    None
                )
                print(f'  ✓ {test["name"]}: Configuration accepted')
            except Exception as e:
                print(f'  ✗ {test["name"]}: {str(e)[:50]}')
                raise
        
        print('Example 4: Mypyc options configuration test passed')
        
except Exception as e:
    print(f'Example 4 failed: {e}')
    raise AssertionError(
        f"hatch_mypyc example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: Compiled Extension Platform Detection

Demonstrates the compiled_extension property for platform-specific shared library detection.

In [ ]:
# Example 5: Compiled extension platform detection
import tempfile
import platform
from pathlib import Path

try:
    from hatch_mypyc.plugin import MypycBuildHook
    
    with tempfile.TemporaryDirectory() as tmpdir:
        root = Path(tmpdir)
        
        # Create project structure
        src_dir = root / "src" / "testpkg"
        src_dir.mkdir(parents=True)
        (src_dir / "__init__.py").write_text('__version__ = "0.1.0"')
        (src_dir / "core.py").write_text('def process(): pass')
        
        # Create build configuration
        config = {
            'include': ['src/**/*.py'],
            'exclude': ['**/__init__.py']
        }
        
        # Initialize build hook
        hook = MypycBuildHook(str(root), config, {}, None, str(root), 'wheel')
        
        print('Platform-specific compilation detection:')
        print(f'  Current platform: {platform.system()}')
        
        # Use compiled_extension property (unique to MypycBuildHook)
        compiled_ext = hook.compiled_extension
        print(f'  Compiled extension: {compiled_ext}')
        
        # Verify platform-specific extension
        if platform.system() == 'Windows':
            expected = '.pyd'
        else:
            expected = '.so'
        
        assert compiled_ext == expected, f'Extension should be {expected} on {platform.system()}'
        print(f'  Expected extension verified: {compiled_ext} matches platform')
        
        # Test file inclusion logic
        test_files = ['src/testpkg/core.py', 'src/testpkg/__init__.py']
        included_files = [f for f in test_files if hook.include_path(f)]
        
        print(f'  Files to compile: {included_files}')
        
        # Simulate artifact naming
        for file in included_files:
            base = file.replace('.py', '')
            artifact_name = f'{base}{compiled_ext}'
            print(f'  Would generate: {artifact_name}')
        
        assert len(included_files) > 0, 'Should find files to compile'
        print('Example 5: Platform detection test passed')
        
except Exception as e:
    print(f'Example 5 failed: {e}')
    raise AssertionError(
        f"hatch_mypyc example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e